In [24]:
%pip install -q -U transformers gradio sentencepiece

In [9]:
%pip install -q --upgrade transformers gradio sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 5.3 MB/s eta 0:00:00


In [10]:
import transformers

print("Transformers version:", transformers.__version__)

from transformers.pipelines import SUPPORTED_TASKS

print("Text-to-text task available:",
      "text2text-generation" in SUPPORTED_TASKS)

Transformers version: 5.18.0
Text-to-text task available: False


In [29]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("AI model loaded successfully!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

AI model loaded successfully!


In [30]:
def chatbot(message, history):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful AI assistant. "
                "Understand the user's entire question before answering. "
                "Answer every part of the question accurately. "
                "Use simple, clear English. "
                "Give at least two complete, meaningful sentences. "
                "Keep answers concise but informative. "
                "Use examples when helpful. "
                "Never give a one-word answer unless explicitly requested. "
                "If a question is unclear, ask for clarification. "
                "Never invent facts."
            )
        }
    ]

    # Include recent conversation history
    for item in (history or [])[-6:]:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")

            if role in ("user", "assistant") and isinstance(content, str):
                messages.append({
                    "role": role,
                    "content": content
                })

        elif isinstance(item, (list, tuple)) and len(item) == 2:
            if item[0]:
                messages.append({
                    "role": "user",
                    "content": str(item[0])
                })
            if item[1]:
                messages.append({
                    "role": "assistant",
                    "content": str(item[1])
                })

    messages.append({
        "role": "user",
        "content": message
    })

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated answer
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    if not answer:
        return (
            "I couldn't generate a suitable answer. "
            "Please try rephrasing your question."
        )

    return answer


demo = gr.ChatInterface(
    fn=chatbot,
    title="EX-10: AI Chatbot",
    description=(
        "Ask questions on different topics. "
        "Get clear, relevant answers in at least two sentences."
    ),
    examples=[
        "What is Artificial Intelligence?",
        "What is Python and where is it used?",
        "Explain Machine Learning with an example.",
        "What is a chatbot?",
        "What is Natural Language Processing?"
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f60befda37763cf3a8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
